# TabFM, explained by example

### Predicting which bank customers are about to leave — without training anything

This notebook assumes **no machine-learning background at all**. Every term is
explained the first time it appears. If something is unclear, that is a fault in
the notebook, not in you.

---

## The problem we are solving

Imagine a spreadsheet of your bank's customers. One row per customer. Columns for
age, country, account balance, how long they have been with you.

Now add one more column: **did this customer leave?**

For your *past* customers, you know the answer — it is written down. For your
*current* customers, you do not. And you would very much like to, because if you
knew who was about to leave, you could try to keep them.

That is the whole task: **fill in a missing column in a spreadsheet.** It is
probably the most common prediction problem in business.

The jargon for it is *tabular prediction* — "tabular" simply meaning "shaped like
a table." The column you are trying to fill in is called the **target**. The other
columns, the ones you use as clues, are called **features**.

---

## The old way, and what it costs

The traditional approach is to **train a model**.

A *model* is just a program that makes predictions. *Training* it means showing it
your past customers — the ones where the answer is already known — and letting it
study them until it works out the patterns. Then you point the trained model at
your current customers and it guesses.

This works. It has been the standard for years. But it carries two costs that
rarely get mentioned:

**1. It needs a great many examples.** You will see this happen live further down.
Given only 25 past customers to learn from, the traditional model in this notebook
does no better than flipping a coin. That is not a bug or a bad setting. The model
starts life knowing absolutely nothing, so 25 examples is genuinely everything it
has, and it is not enough to justify any conclusion.

**2. It needs an expert to set it up.** These programs cannot read text — the word
`"Germany"` is meaningless to them, so a person has to convert every text column
into numbers. Blank cells make them fail, so a person has to decide what to put in
the gaps. And they have dozens of settings to choose. Getting any of this subtly
wrong produces a model that looks fine and is quietly useless.

---

## What TabFM does differently

Google took one program and showed it **hundreds of millions of spreadsheets** —
not yours, just an enormous variety of invented ones. It was not learning about
banks, or customers, or churn. It was learning something more general: *what
patterns in spreadsheets tend to look like.*

That program is now finished. Its knowledge is frozen. It will never learn
anything again — including from you.

So when you hand it your customer table, it does not study your data the way the
old approach does. It reads your handful of known customers, effectively thinks
*"I have seen a shape like this before,"* and applies that recognition to the
customers you are asking about. All in one pass, in seconds.

> ### The analogy worth holding on to
>
> The traditional model is a **brand-new hire on their first day**. They know
> nothing, and you must teach them your business from scratch. Show them 25
> examples and they will learn nothing useful.
>
> TabFM is a **veteran analyst who has read a million spreadsheets** across a
> hundred industries. Show that veteran 25 rows and they will spot the pattern —
> not because they know your bank, but because they know how patterns behave.
>
> This is the entire reason TabFM wins when data is scarce. It brings prior
> experience to the table. The new hire has none.

The technical name for reading examples and answering in one go, without ever
updating what you know, is **in-context learning**. It is the same trick that lets
a chatbot do a task you only showed it three examples of.

---

## The question this notebook actually answers

Not *"does TabFM work?"* — that is easy and not very useful.

The real question is: **when should I use it instead of the traditional approach?**

We answer it by running a fair race. We give every method 25 customers to learn
from, then 50, then 100, and so on up to thousands, and we watch how each one
improves. Where one method overtakes the other is your decision rule.

---

## Before you run anything

Go to **Runtime → Change runtime type → T4 GPU**, then come back.

A GPU is specialised hardware that does this kind of arithmetic enormously faster.
It is free in Colab. Without it this notebook still works, but TabFM slows from
seconds to many minutes, and you will lose patience.

With a GPU, expect **10 to 20 minutes** in total. Most of that is a one-off
download of the TabFM program itself, which is a hefty 6.6 gigabytes.

---
# Part 1 — Setup

## 1.1 Check you really have a GPU

Worth checking explicitly. A Colab session can quietly fall back to the slow
hardware, and you would just think TabFM was disappointing.

In [ ]:
import subprocess

out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)

if out.returncode == 0 and out.stdout.strip():
    print("GPU found:", out.stdout.strip())
    print("You are good to go.")
else:
    print("NO GPU FOUND.")
    print("Go to  Runtime > Change runtime type > T4 GPU,  then run this cell again.")
    print("(The notebook still works without one, but TabFM will be very slow.)")

## 1.2 Install TabFM

**One trap to know about.** There is a `tabfm` package on the usual Python
software repository, but installing it the usual way is known to fail: it looks for
a file whose name does not match what Google actually published, so you download
several gigabytes and *then* get an error. The cell below therefore installs
straight from Google's own code repository instead.

Two things to expect while it runs:

- It takes a few minutes. That is normal.
- **Colab may ask you to restart the runtime** when it finishes, because TabFM
  wants a specific version of one of its dependencies. If a "Restart session"
  button appears, click it, then carry on from section 1.3 below. You do **not**
  need to run the install again.

In [ ]:
%pip install -q "tabfm[pytorch] @ git+https://github.com/google-research/tabfm" safetensors

## 1.3 Check the install worked

In [ ]:
import tabfm
from tabfm import TabFMClassifier, TabFMRegressor

print("TabFM imported successfully.")
print()
print("The library gives you two tools:")
print("  TabFMClassifier - predicts a CATEGORY   (will they leave: yes or no?)")
print("  TabFMRegressor  - predicts a NUMBER     (how much will they spend?)")
print()
print("This notebook uses the first one.")

---
# Part 2 — The customer data

We build a table of 12,000 bank customers. It is invented rather than downloaded,
which means the notebook needs no external files and gives you the same answer
every time you run it.

But it is invented to be **awkward in the ways real business data is awkward**.
That matters. A tidy dataset where one column gives the answer away would make
every method look identical, and you would learn nothing.

Here is what makes it realistic:

**Text columns sit next to numbers.** `country` contains words like `"Germany"`;
`balance` contains numbers. Most prediction programs cannot read the words.

**Some cells are blank.** Some balances and some plan tiers were simply never
recorded, exactly as in any real customer database.

**The patterns are not simple straight lines.** Churn risk is *U-shaped* in age:
the very young and the near-retirement customers leave most, while the
middle-aged stay. And it is *non-monotonic* in product count, meaning it does not
just rise or just fall — holding two products is the loyal sweet spot, while
holding one is risky and holding four is riskier still.

**Two clues only make sense in combination.** A large balance predicts churn *only
when the customer is also inactive*. Customers acquired through a partner churn
more *only during their first two years*. Neither pattern is visible if you look
at any single column on its own — you have to see two at once. These are called
**interaction effects**, and they are what separates a method that genuinely
understands a table from one that does not.

In [ ]:
import numpy as np
import pandas as pd

# Which columns hold words, and which hold numbers. We will need this later.
CATEGORICAL = ["country", "gender", "plan_tier", "acquisition_channel"]
NUMERIC = ["age", "tenure_years", "balance", "num_products", "has_credit_card",
           "is_active_member", "estimated_salary", "support_tickets",
           "months_since_last_login"]


def make_churn_data(n_rows=12_000, seed=0):
    rng = np.random.default_rng(seed)

    # ---------- the customer characteristics ----------
    country = rng.choice(["France", "Germany", "Spain"], n_rows, p=[0.5, 0.25, 0.25])
    gender = rng.choice(["Male", "Female"], n_rows)
    plan_tier = rng.choice(["Basic", "Plus", "Premium"], n_rows, p=[0.55, 0.3, 0.15])
    channel = rng.choice(["branch", "online", "partner", "referral"], n_rows,
                         p=[0.3, 0.4, 0.2, 0.1])

    age = np.clip(rng.normal(39, 11, n_rows), 18, 88).round(0)
    tenure = np.clip(rng.gamma(2.0, 2.2, n_rows), 0, 22).round(1)
    # Plenty of customers sit at exactly zero balance -- a realistic lump.
    balance = np.where(rng.random(n_rows) < 0.28, 0.0,
                       np.abs(rng.normal(78_000, 45_000, n_rows))).round(2)
    num_products = rng.choice([1, 2, 3, 4], n_rows, p=[0.45, 0.4, 0.11, 0.04])
    has_cc = rng.binomial(1, 0.7, n_rows)
    active = rng.binomial(1, 0.52, n_rows)
    salary = np.clip(rng.normal(62_000, 26_000, n_rows), 12_000, None).round(2)
    tickets = rng.poisson(0.6, n_rows)
    months_idle = np.clip(rng.exponential(2.4, n_rows), 0, 36).round(1)

    # ---------- the hidden rule that decides who leaves ----------
    # 'z' is a churn-risk score. Higher means more likely to leave.
    z = -3.10                                                # baseline: most people stay

    z += 0.9 * ((age - 44) / 15.0) ** 2                      # U-shaped in age
    z += np.select([num_products == 1, num_products == 2,    # 2 products = sweet spot
                    num_products == 3, num_products >= 4],
                   [0.55, -0.75, 0.85, 1.7])
    z += 1.15 * (1 - active)                                 # inactive members leave
    z += 0.62 * (country == "Germany")                       # a market effect
    z += 0.45 * tickets                                      # complaints drive churn
    z += 0.40 * (months_idle > 6)                            # a cliff edge, not a slope
    z -= 0.09 * np.minimum(tenure, 8)                        # loyalty stops helping at 8y
    z += 0.35 * (plan_tier == "Basic")
    z -= 0.30 * (plan_tier == "Premium")

    z += 0.85 * ((balance > 120_000) & (active == 0))        # INTERACTION
    z += 0.70 * ((channel == "partner") & (tenure < 2))      # INTERACTION

    # Real life is not deterministic: people leave for reasons no column records.
    z += rng.normal(0, 0.55, n_rows)

    # Turn the risk score into an actual yes/no outcome.
    churn = rng.binomial(1, 1 / (1 + np.exp(-z)))

    df = pd.DataFrame({
        "age": age, "tenure_years": tenure, "balance": balance,
        "num_products": num_products, "has_credit_card": has_cc,
        "is_active_member": active, "estimated_salary": salary,
        "support_tickets": tickets, "months_since_last_login": months_idle,
        "country": country, "gender": gender, "plan_tier": plan_tier,
        "acquisition_channel": channel,
    })

    # Gaps in the records, as in any real database.
    df.loc[rng.random(n_rows) < 0.05, "balance"] = np.nan
    df.loc[rng.random(n_rows) < 0.03, "plan_tier"] = None

    return df, pd.Series(churn, name="churned")


X_all, y_all = make_churn_data()

print(f"{len(X_all):,} customers")
print(f"{y_all.mean():.1%} of them churned (left the bank)")
print(f"{X_all.balance.isna().sum()} blank balances, "
      f"{X_all.plan_tier.isna().sum()} blank plan tiers")
X_all.head()

Look closely at the column types below.

`country`, `gender`, `plan_tier` and `acquisition_channel` hold **raw text**.
`balance` holds some **blanks** (shown as `NaN`, which stands for "not a number").

Hold on to that, because shortly TabFM will eat this table exactly as it stands,
while the traditional method will refuse to touch it until we have cleaned it up.

In [ ]:
X_all.dtypes.to_frame("what this column holds")

## Splitting the data

We set aside **3,000 customers as a test set** and never let any method learn from
them. Every score in this notebook is measured on those same 3,000 people, so the
comparison is fair.

The other 9,000 form a **pool**. This is where we will draw training examples from
— first 25 of them, then 50, then 100, and so on.

Why hold data back at all? Because a model marking its own homework always scores
brilliantly. The only meaningful question is how it performs on customers it has
never seen.

In [ ]:
X_test, y_test = X_all.iloc[-3000:].reset_index(drop=True), y_all.iloc[-3000:].reset_index(drop=True)
X_pool, y_pool = X_all.iloc[:-3000].reset_index(drop=True), y_all.iloc[:-3000].reset_index(drop=True)

print(f"Pool to draw training examples from : {len(X_pool):,} customers")
print(f"Test set, fixed and never learned from: {len(X_test):,} customers")

---
# Part 3 — How we keep score

Before comparing anything, we have to agree on what "good" means. This section
matters more than it looks, and it is worth reading slowly.

## Why not just count how many predictions were right?

The obvious score is **percentage correct**: out of 3,000 customers, how many did
we call correctly?

It is a trap, and here is why. Only about **24%** of customers churn. So consider
the laziest possible model — one that ignores every column and always predicts
*"this customer will stay."* It never finds a single churner. It is completely,
utterly useless.

And yet it gets **about 76% correct**, because it is right about everyone who
stays. It sounds like a solid B grade. It is worth nothing at all.

This is called the **class imbalance** problem: when one answer is much more
common than the other, percentage-correct rewards you for ignoring the rare case —
which is invariably the case you actually care about.

## What we use instead: ROC AUC

**ROC AUC** has an intimidating name and a genuinely simple meaning:

> **Pick one customer who left and one customer who stayed, at random. How often
> does the model consider the leaver riskier than the stayer?**

That is the whole definition. It is a percentage, written as a decimal:

| ROC AUC | What it means |
|---|---|
| **0.50** | Coin flip. The model has no idea. |
| **0.65** | Weak, but real signal. |
| **0.75** | Genuinely useful in practice. |
| **0.90** | Very strong. |
| **1.00** | Perfect. Ranks every leaver above every stayer. |

Two properties make it the right choice here:

**It cannot be fooled by the lazy model.** If you predict the same thing for
everybody, you rank nobody above anybody, and you score exactly 0.50. The lazy
model's 76% evaporates the moment you measure it honestly.

**It scores the ranking, not a yes/no verdict.** In real life you do not act on
all 3,000 customers — you have budget to phone maybe 200. What you need is the
*right 200 at the top of the list*. ROC AUC measures exactly that ability to sort
people by risk, which is what the business actually uses.

The cell below demonstrates all of this rather than asking you to take my word.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

# The lazy model: predict "stays" for absolutely everyone.
lazy = np.zeros(len(y_test))

print("THE LAZY MODEL - always predicts 'this customer will stay'")
print(f"   percentage correct : {accuracy_score(y_test, lazy):>6.1%}   <- looks respectable!")
print(f"   ROC AUC            : {roc_auc_score(y_test, lazy):>6.3f}   <- honest: it knows nothing")
print(f"   churners it found  : 0 out of {int(y_test.sum())}")
print()
print("Percentage correct gives a useless model a passing grade.")
print("ROC AUC gives it exactly 0.500, which is the truth.")

### Let us prove the definition is really that simple

The next cell takes the plain-English definition literally: pick a random churner
and a random stayer, 200,000 times, and count how often the model ranks the
churner higher. If the definition above is right, that percentage should match the
ROC AUC almost exactly.

We use each customer's *true* risk score as a stand-in for a very good model.

In [ ]:
rng = np.random.default_rng(0)
churners = np.flatnonzero(y_test == 1)
stayers  = np.flatnonzero(y_test == 0)

# A near-perfect scorer, for demonstration: a little noise on the true answer.
demo_score = y_test.to_numpy() + rng.normal(0, 0.45, len(y_test))

a, b = rng.choice(churners, 200_000), rng.choice(stayers, 200_000)
head_to_head = (demo_score[a] > demo_score[b]).mean()

print(f"Random churner ranked above random stayer: {head_to_head:.1%} of the time")
print(f"ROC AUC computed by the library          : {roc_auc_score(y_test, demo_score):.3f}")
print()
print("Same number. That IS what ROC AUC measures -- nothing more mysterious than that.")

## One more thing: what is the best score anyone could get?

A score of 0.75 means nothing unless you know what perfection looks like on *this*
problem. And perfection is **not** 1.00 here.

We invented this data, so we know the exact rule that decides who leaves — and we
deliberately included randomness in it, because in real life people leave for
reasons no spreadsheet records. Nobody can predict a coin flip.

So the cell below computes the **ceiling**: the score achieved by an imaginary
oracle that knows the true rule perfectly. No method in this notebook can beat it.
That number is the yardstick for everything that follows.

In [ ]:
# Rebuild the exact hidden rule, this time WITHOUT the random part.
_r = np.random.default_rng(0)
_country = _r.choice(["France", "Germany", "Spain"], 12_000, p=[0.5, 0.25, 0.25])
_gender = _r.choice(["Male", "Female"], 12_000)
_plan = _r.choice(["Basic", "Plus", "Premium"], 12_000, p=[0.55, 0.3, 0.15])
_chan = _r.choice(["branch", "online", "partner", "referral"], 12_000, p=[0.3, 0.4, 0.2, 0.1])
_age = np.clip(_r.normal(39, 11, 12_000), 18, 88).round(0)
_ten = np.clip(_r.gamma(2.0, 2.2, 12_000), 0, 22).round(1)
_bal = np.where(_r.random(12_000) < 0.28, 0.0, np.abs(_r.normal(78_000, 45_000, 12_000))).round(2)
_np_ = _r.choice([1, 2, 3, 4], 12_000, p=[0.45, 0.4, 0.11, 0.04])
_cc = _r.binomial(1, 0.7, 12_000)
_act = _r.binomial(1, 0.52, 12_000)
_sal = np.clip(_r.normal(62_000, 26_000, 12_000), 12_000, None).round(2)
_tick = _r.poisson(0.6, 12_000)
_idle = np.clip(_r.exponential(2.4, 12_000), 0, 36).round(1)

_z = -3.10
_z += 0.9 * ((_age - 44) / 15.0) ** 2
_z += np.select([_np_ == 1, _np_ == 2, _np_ == 3, _np_ >= 4], [0.55, -0.75, 0.85, 1.7])
_z += 1.15 * (1 - _act) + 0.62 * (_country == "Germany") + 0.45 * _tick
_z += 0.40 * (_idle > 6) - 0.09 * np.minimum(_ten, 8)
_z += 0.35 * (_plan == "Basic") - 0.30 * (_plan == "Premium")
_z += 0.85 * ((_bal > 120_000) & (_act == 0))
_z += 0.70 * ((_chan == "partner") & (_ten < 2))

# Average over the randomness the oracle cannot see.
_draws = np.random.default_rng(7).normal(0, 0.55, (400, 12_000))
oracle = (1 / (1 + np.exp(-(_z + _draws)))).mean(axis=0)[-3000:]

CEILING = roc_auc_score(y_test, oracle)
print(f"THE CEILING: {CEILING:.3f} ROC AUC")
print()
print("An oracle knowing the exact churn rule scores this. Nothing can beat it,")
print("because the rest is genuine randomness -- people leaving for reasons no")
print("column records.")
print()
print("So read every score below against this scale:")
print(f"   0.500 = knows nothing")
print(f"   {CEILING:.3f} = knows everything knowable")

---
# Part 4 — TabFM, with only 50 customers to learn from

Now the part worth pausing on.

We give TabFM **50 customers** — that is all — and ask it to rank all 3,000 test
customers by churn risk.

The raw table goes straight in. Text columns and blank cells included. No cleaning,
no conversion, no settings to choose.

## First, load it

The next cell downloads the pretrained program from Google (about 6.6 GB, once per
session) and gets it ready.

In [ ]:
import inspect
import time
import torch
from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE}   ('cuda' means the fast GPU, 'cpu' means the slow path)")
print("Downloading the model if this is the first run. Please be patient...")

try:
    tabfm_model = tabfm_v1_0_0.load(model_type="classification")
except TypeError:
    tabfm_model = tabfm_v1_0_0.load()   # older versions take no argument

print("Ready.")

TabFM's optional settings vary a little between releases, so instead of guessing,
the next cell simply asks your installed copy what it accepts. Anything it lists is
a dial you *could* turn — though the entire point of TabFM is that you do not have
to touch any of them.

In [ ]:
print("Optional settings your version of TabFMClassifier accepts:")
for name, p in inspect.signature(TabFMClassifier.__init__).parameters.items():
    if name != "self":
        print(f"    {name:22} (default: {p.default})")

In [ ]:
_ACCEPTED = set(inspect.signature(TabFMClassifier.__init__).parameters)


def tabfm_predict(X_train, y_train, X_score):
    # Show TabFM some known customers, then ask it to rank unknown ones.
    # Returns each scored customer's churn probability, from 0.0 to 1.0.
    kwargs = {"model": tabfm_model}
    if "device" in _ACCEPTED:
        kwargs["device"] = DEVICE

    clf = TabFMClassifier(**kwargs)
    try:
        clf.fit(X_train, y_train.to_numpy())
    except (ValueError, TypeError):
        # Safety net, in case this build dislikes blanks in the text columns.
        fill = lambda d: d.assign(**{c: d[c].fillna("unknown") for c in CATEGORICAL})
        clf = TabFMClassifier(**kwargs)
        clf.fit(fill(X_train), y_train.to_numpy())
        X_score = fill(X_score)

    return clf.predict_proba(X_score)[:, 1]

## Now the actual prediction

Three lines of real work. Watch what goes in: `X_small` is the raw table, text and
blanks and all.

In [ ]:
X_small, y_small = X_pool.iloc[:50], y_pool.iloc[:50]

start = time.perf_counter()
risk_scores = tabfm_predict(X_small, y_small, X_test)
seconds = time.perf_counter() - start

score = roc_auc_score(y_test, risk_scores)

print(f"Learned from : {len(X_small)} customers")
print(f"Ranked       : {len(X_test):,} customers")
print(f"Time taken   : {seconds:.1f} seconds")
print()
print(f"ROC AUC      : {score:.3f}")
print(f"   (0.500 = knows nothing, {CEILING:.3f} = the ceiling)")
print(f"   That is {(score - 0.5) / (CEILING - 0.5):.0%} of the way from useless to perfect,")
print(f"   having seen 50 customers.")

## What `fit()` actually did — and did not do

This is the conceptual heart of the notebook.

`clf.fit(X, y)` did **not** change a single one of TabFM's internal settings. They
are frozen. They are identical to those of every other TabFM user on earth.

All `fit` did was **remember your 50 rows**, plus some trivial bookkeeping.

The real work happened in the *next* step. There, your 50 known customers and the
3,000 unknown ones were fed through the program **together, as a single input**.
It read the known examples, recognised the pattern, and applied it — in one pass.

Three consequences follow, and they explain almost all of TabFM's behaviour:

**1. There is no trained model to save.** Your training rows *are* the model. They
must travel with you to every prediction.

**2. Predicting is slow.** Every single prediction re-reads all your training rows.
A traditional model does its hard work once, up front, then answers instantly
forever. TabFM does the work every time.

**3. Your training data cannot grow without limit.** It all has to fit inside the
program's *context window* — the maximum amount it can read at once. This is why
very large tables must be cut down to a sample first. It is also why the experiment
later stops at 2,000 rows for TabFM but keeps going for the traditional method.

---
# Part 5 — The traditional approach, and the work it demands

Time to line up the competition. Two traditional methods:

**Gradient boosting.** The long-standing champion for spreadsheet data, and TabFM's
real rival. It builds hundreds of small decision trees, each one correcting the
mistakes of the ones before it. A *decision tree* is just a flowchart of yes/no
questions — "is the customer inactive? is the balance above 120,000?" — ending in a
prediction.

**Logistic regression.** A deliberately simple method that draws one straight line
through the data. It is here as an honest yardstick, because simple methods often
*beat* sophisticated ones when data is very scarce. There is less to get wrong.

## Notice how much scaffolding they need

Read the cell below and compare it to the three lines TabFM needed. Before either
method can see the data at all, we must:

- **fill in the blank cells** (an "imputer"), because they cannot handle gaps;
- **turn every text column into numbers** (a "one-hot encoder"), because they
  cannot read words;
- **rescale the numeric columns** so that salary, measured in tens of thousands,
  does not drown out age, measured in tens.

*That* is the feature engineering TabFM removes. Every line of it is a chance to
introduce a silent mistake — and this is a short, tidy example.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_traditional_models():
    # --- all the cleaning logistic regression needs before it can start ---
    cleaning = ColumnTransformer([
        ("numbers", Pipeline([
            ("fill_blanks", SimpleImputer(strategy="median")),
            ("rescale", StandardScaler()),
        ]), NUMERIC),
        ("words", Pipeline([
            ("fill_blanks", SimpleImputer(strategy="most_frequent")),
            ("text_to_numbers", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL),
    ])
    logistic = Pipeline([("clean", cleaning),
                         ("model", LogisticRegression(max_iter=2000))])

    # --- gradient boosting copes with blank numbers, but still cannot read words ---
    boosting = Pipeline([
        ("clean", ColumnTransformer([
            ("numbers", "passthrough", NUMERIC),
            ("words", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
        ])),
        ("model", HistGradientBoostingClassifier(random_state=0)),
    ])

    return {"Gradient boosting": boosting, "Logistic regression": logistic}


print("Same 50 customers, given to the traditional methods:")
print()
for name, model in build_traditional_models().items():
    model.fit(X_small, y_small)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    print(f"   {name:22} ROC AUC {auc:.3f}")

print()
print(f"   {'TabFM (from above)':22} ROC AUC {score:.3f}")

---
# Part 6 — The experiment that answers "when should I use this?"

A single comparison at one training size proves very little — it could be luck. So
we do it properly.

We sweep the amount of training data across two orders of magnitude, from 25
customers up to 8,000, and repeat each size with three different random samples so
we are measuring a real effect rather than a lucky draw.

## What to expect, and why

**On the left, where data is scarce, TabFM should lead comfortably.** It arrives
already knowing what patterns look like. Gradient boosting arrives knowing nothing
and has almost nothing to work with — expect it to be close to a coin flip at 25
customers.

**On the right, gradient boosting should close the gap and overtake.** Given enough
examples it needs no prior experience, because your own data tells it everything.
The veteran's advantage was never the ceiling — it was the head start.

**Somewhere the lines cross. That crossing point is your decision rule.**

## Why TabFM stops at 2,000

Look at `TABFM_SIZES` below: it stops at 2,000, while the traditional methods carry
on to 8,000. That is not an oversight — it is the context-window limit from Part 4
made concrete. TabFM must carry every training row through the program on every
prediction, so both time and memory grow with the training set. The traditional
methods have no such constraint.

This is itself one of the most practical differences between the two approaches.

---

**Runtime:** roughly 10 to 20 minutes on a GPU. Set `QUICK = True` just below for a
rough version in about 3 minutes — fewer sizes, and one sample each instead of
three. The shape of the answer comes out the same; it is just noisier.

In [ ]:
QUICK = False          # <-- set to True for a ~3 minute version

TABFM_SIZES       = [25, 50, 100, 250, 500, 1000, 2000]
TRADITIONAL_ONLY  = [4000, 8000]        # beyond TabFM's reach
SEEDS             = [0, 1, 2]

if QUICK:
    TABFM_SIZES, TRADITIONAL_ONLY, SEEDS = [25, 100, 500, 2000], [8000], [0]

records = []

for n in TABFM_SIZES + TRADITIONAL_ONLY:
    for seed in SEEDS:
        picked = np.random.default_rng(seed).choice(len(X_pool), n, replace=False)
        X_train, y_train = X_pool.iloc[picked], y_pool.iloc[picked]

        if y_train.nunique() < 2:
            continue    # a tiny sample can accidentally contain only one answer

        if n in TABFM_SIZES:
            t0 = time.perf_counter()
            auc = roc_auc_score(y_test, tabfm_predict(X_train, y_train, X_test))
            records.append({"customers": n, "seed": seed, "method": "TabFM (zero-shot)",
                            "auc": auc, "seconds": time.perf_counter() - t0})

        for name, model in build_traditional_models().items():
            t0 = time.perf_counter()
            model.fit(X_train, y_train)
            auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
            records.append({"customers": n, "seed": seed, "method": name,
                            "auc": auc, "seconds": time.perf_counter() - t0})

    so_far = pd.DataFrame(records).query("customers == @n").groupby("method").auc.mean()
    print(f"{n:>5} customers | " + "  ".join(f"{m}: {v:.3f}" for m, v in so_far.items()))

results = pd.DataFrame(records)
print("\nExperiment complete.")

## The results

The table first, because it is the exact record. The picture comes after.

Read every number against the two anchors: **0.500 is knowing nothing**, and the
**ceiling** computed in Part 3 is knowing everything knowable.

In [ ]:
scores = results.pivot_table(index="customers", columns="method", values="auc").round(3)
scores.index.name = "training customers"
print(f"ROC AUC   (0.500 = useless, {CEILING:.3f} = the ceiling)")
print("Blank cells for TabFM: beyond its context-window limit.\n")
scores

In [ ]:
timing = results.pivot_table(index="customers", columns="method", values="seconds").round(2)
timing.index.name = "training customers"
print("Seconds to learn and then rank all 3,000 test customers:\n")
timing

## The picture

In [ ]:
import matplotlib.pyplot as plt

STYLE = {
    "TabFM (zero-shot)":   {"color": "#2a78d6", "marker": "o"},
    "Gradient boosting":   {"color": "#eb6834", "marker": "s"},
    "Logistic regression": {"color": "#1baf7a", "marker": "^"},
}

fig, ax = plt.subplots(figsize=(9.5, 5.8), dpi=130)
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

line_ends = []
for name, style in STYLE.items():
    series = scores[name].dropna()
    ax.plot(series.index, series.values, linewidth=2, markersize=8,
            color=style["color"], marker=style["marker"], label=name,
            markeredgecolor="#fcfcfb", markeredgewidth=1.5, zorder=3)
    line_ends.append([series.values[-1], series.index[-1], name, style["color"]])

# The two anchors that give every number meaning.
ax.axhline(CEILING, color="#7a7972", linewidth=1.2, linestyle=(0, (5, 3)), zorder=1)
ax.annotate(f"ceiling {CEILING:.3f} — nothing can do better",
            (TABFM_SIZES[0], CEILING), xytext=(0, 6), textcoords="offset points",
            color="#52514e", fontsize=8.5)

ax.axhline(0.5, color="#a8a79e", linewidth=1, linestyle=(0, (4, 4)), zorder=1)
ax.annotate("0.500 — pure guesswork", (TABFM_SIZES[0], 0.5), xytext=(0, 6),
            textcoords="offset points", color="#7a7972", fontsize=8.5)

# Direct labels, nudged apart if two lines finish close together, so that
# identity never depends on colour alone.
top, bottom = max(CEILING, scores.max().max()), min(0.5, scores.min().min())
gap = (top - bottom) * 0.06
line_ends.sort()
for i in range(1, len(line_ends)):
    if line_ends[i][0] - line_ends[i - 1][0] < gap:
        line_ends[i][0] = line_ends[i - 1][0] + gap

right = max(scores.index)
for label_y, end_x, name, colour in line_ends:
    ax.annotate(name, xy=(end_x, scores[name].dropna().values[-1]),
                xytext=(right * 1.25, label_y), color=colour, fontsize=9.5, va="center",
                arrowprops=dict(arrowstyle="-", color=colour, linewidth=0.8,
                                shrinkA=0, shrinkB=2, alpha=0.5))

ax.set_xscale("log")
all_sizes = sorted(scores.index)
ax.set_xticks(all_sizes)
ax.set_xticklabels([f"{s:,}" for s in all_sizes])
ax.set_xlim(all_sizes[0] * 0.85, right * 4.2)
ax.set_xlabel("How many customers the method was allowed to learn from  (log scale)",
              fontsize=10, color="#52514e")
ax.set_ylabel("ROC AUC on the same 3,000 test customers", fontsize=10, color="#52514e")
ax.set_title("When is TabFM worth it? Accuracy against how much data you have",
             fontsize=13.5, color="#0b0b0b", pad=14, loc="left")

ax.grid(axis="y", color="#e6e5e0", linewidth=1, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#d8d7d1")
ax.tick_params(colors="#52514e", labelsize=9)
ax.legend(frameon=False, loc="lower right", fontsize=9.5)

plt.tight_layout()
plt.show()

---
# Part 7 — How to read your chart

Your exact numbers will shift a little with the random seed. Look for these four
things.

**1. The left edge is TabFM's home ground.** With 25 or 50 customers, gradient
boosting is near-useless — quite possibly a flat 0.500, a pure coin flip. Again,
this is not misconfiguration. With 25 examples there is not enough evidence to
justify splitting the data even once, so it declines to commit and predicts much
the same thing for everybody. TabFM has no such problem, because it is not learning
churn from scratch. It is recognising a shape it has seen before.

**2. The lines converge, and probably cross.** As customers accumulate, gradient
boosting climbs steadily and TabFM flattens out. Prior experience matters less and
less once your own data can tell the whole story. **Where they cross is your
decision rule** — below that many labelled rows, reach for TabFM; above it, reach
for gradient boosting.

**3. The right-hand end shows TabFM's real limit.** The blue line simply stops at
2,000 while the others continue. In practice that is often the deciding factor: if
you have 50,000 labelled rows, TabFM cannot use them all, and a method that can
will win.

**4. Look at the timing table again.** Gradient boosting fits in well under a
second. TabFM takes far longer, and gets **slower as it gets better**, because
every training row rides along on every prediction. Every other method here gets
cheaper to use as it improves. TabFM does not.

---
# Part 8 — The practical summary

## Use TabFM when…

**You have very little labelled data.** Hundreds of rows, not millions. This is by
far the strongest reason, and it is what the left of the chart shows. It is also
extremely common: labelling data is expensive, and most real projects start with a
few hundred examples someone assembled by hand.

**You want an answer today.** A credible result in ten minutes with no setup tells
you quickly whether your data contains any usable signal at all — before anyone
commits budget to a proper project.

**Nobody available is a machine-learning specialist.** No dials means far fewer
ways to be quietly wrong. Badly configured data cleaning is a common failure and
almost invisible when it happens.

**You have many small tables rather than one big one.** Building and maintaining a
tuned model for each is expensive. One frozen program serving all of them is not.

**Your data is messy.** Text columns and blank cells go in untouched.

## Stick with the traditional approach when…

**You have plenty of labelled data.** Past a few thousand rows the advantage fades,
and a well-tuned gradient boosting model is very hard to beat.

**Predictions must be fast or cheap.** TabFM needs a GPU and is thousands of times
slower per prediction. If you are scoring millions of rows, or answering in
milliseconds inside a web page, it is the wrong tool.

**You must explain individual decisions.** A bank refusing a loan has to say why.
Tree-based methods expose their reasoning as a readable flowchart. TabFM's is
buried inside a single opaque pass.

**You are building a commercial product.** See the licence note below — this one is
a hard stop, not a trade-off.

## The limits, in plain terms

| Limit | What it means for you |
|---|---|
| **Maximum 10 categories** | Fine for yes/no, or "which of five plans". Impossible for "which of our 300 products" — and this is built into the design, not a setting you can raise. |
| **Up to about 500 columns** | Generous. Most business tables have well under 50. |
| **Bounded training rows** | Everything must fit in one read. Large tables must be sampled down. |
| **Tables only** | Not images, audio, video, or free-form text. |
| **About 6.6 GB to download** | Once per session in Colab. |
| **Needs Python 3.11+** | Colab is fine. Some older cloud environments are not. |

## The licence catch — please read this one

The **code** is open source (Apache 2.0). The **pretrained knowledge is not.**

It ships under the *TabFM Non-Commercial License v1.0*, restricting use to
non-commercial, non-production purposes. In plain terms: wonderful for learning,
research, and internal experiments; you **cannot** put it inside a product you
sell. Google has said BigQuery integration is coming, which is the likely route to
a commercially usable version.

## Where TabFM sits in the wider picture

TabFM is not alone — it is Google's entry in a fast-moving field. **TabPFN** from
Prior Labs pioneered this approach and **TabICL** is another. TabFM is also the
table-shaped sibling of **TimesFM**, Google's zero-shot model for time series.

The shared idea is the genuinely interesting part: train once on a vast quantity of
invented data, then solve brand-new problems by *reading examples*, rather than by
training all over again.

---

# Things to try next

**1. Find your exact crossing point.** Add more sizes between 500 and 4,000. That
crossing point is the decision rule for data shaped like this.

**2. Use your own spreadsheet.** Load it with `pd.read_csv("yourfile.csv")`, list
your text columns in `CATEGORICAL` and your numeric ones in `NUMERIC`, name your
target column, and rerun. This is the fastest way to learn whether your own problem
has signal in it — which is genuinely useful information, whatever you build later.

**3. Predict a number instead of a category.** Swap `TabFMClassifier` for
`TabFMRegressor` and load it with `load(model_type="regression")` to predict, say,
customer lifetime value.

**4. Make the problem harder and watch the gap widen.** In `make_churn_data`,
change the noise from `0.55` to `1.2`. Prior experience matters *more* when the
signal is weaker, so TabFM's small-data lead should grow. This also lowers the
ceiling — rerun the ceiling cell to see by how much.